# 🧴 A/B Testing — La Redoma Rosa 🫧
## ¿Qué estética aplicamos a una página web dedicada a la venta de productos naturales?


**La Redoma Rosa** es una tienda online asturiana de productos elaborados con ingredientes naturales y recetas tradicionales. Pertenece a una empresaria arraigada en Asturias, con negocios tales como librerías y tostadores de café.

---

<img src="../imagenes/pato_Anaïs.png"  width="420"/>

---
El equipo de diseño propone cambiar la estética de la web. Antes de lanzarlo al 100% de los usuarios, hacemos un **A/B test** para decidir con datos, no con opiniones.

---

<img src="../imagenes/opciones_A_B.png" width="620"/>

---

| Versión | Estética | Descripción |
|---------|----------|-------------|
| **A** | Vintage Rosa | Acogedora, retro, evoca naturalidad y tradición |
| **B** | Moderna | Limpia, tecnológica, fría |

**Hipótesis:** La estética vintage (A) generará más conversiones porque transmite los valores de la marca — naturalidad, autenticidad y confianza en los métodos tradicionales.

**Métrica principal:** Tasa de conversión — porcentaje de visitantes que realizan una compra.

---

### Pasos de la práctica

| Paso | Qué haremos |
|------|-------------|
| **1** | Cargar y explorar el dataset |
| **2** | Calcular las métricas por versión |
| **3** | Visualizar los resultados |
| **4** | Test estadístico: ¿es real la diferencia? |
| **5** | Conclusión y decisión |


---
## Paso 1: Cargar y explorar el dataset

Empezamos instalando   
- `statsmodels`: librería diseñada para realizar análisis estadísticos, pruebas de hipótesis e intervalos de confianza específicamente para datos de proporciones.    
- `maplotlib` para los gráficos.


In [ ]:
! pip install statsmodels matplotlib

Continuamos importando las librerías y cargando los datos.  
Antes de analizar nada, siempre dedicamos unos minutos a **entender qué tenemos**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportions_ztest
# import warnings
# warnings.filterwarnings('ignore')

# Cargamos el dataset
# 👉 Cambia la ruta si tienes el CSV en una subcarpeta
df = pd.read_csv('../dataset/la_redoma_rosa.csv')

print(f"Dataset cargado correctamente")
print(f"Total de registros: {len(df):,}")


### ¿Qué columnas tenemos?

In [ ]:
df.head(8)


In [ ]:
# Descripción de cada columna:
# usuario_id        → identificador único de cada visitante
# version           → A (vintage rosa) o B (moderna)
# pagina            → nombre de la versión que vio
# concejo           → concejo asturiano del visitante
# producto_visto    → producto que estaba mirando
# tiempo_sesion_seg → segundos que estuvo en la web
# convirtio         → 1 si compró, 0 si no compró

print("Resumen del dataset:")
print(df.info())


In [ ]:
# ¿Cuántos usuarios hay en cada versión?
print("Usuarios por versión:")
print(df['version'].value_counts())
print()

# ¿De qué concejos son?
print("Top 5 concejos:")
print(df['concejo'].value_counts().head())
print()

# ¿Cuántos compraron en total?
print(f"Total conversiones: {df['convirtio'].sum():,}")
print(f"Tasa global: {df['convirtio'].mean():.2%}")


---
## Paso 2: Calcular las métricas por versión

Calculamos la **tasa de conversión** de cada grupo: ¿qué porcentaje de visitantes de cada versión terminó comprando?


In [ ]:
# Agrupamos por versión y calculamos
resumen = df.groupby('version').agg(
    visitantes   = ('convirtio', 'count'),
    conversiones = ('convirtio', 'sum'),
    tasa         = ('convirtio', 'mean')
).reset_index()

resumen['tasa_%'] = (resumen['tasa'] * 100).round(2)

print("=== Métricas por versión ===")
display(resumen[['version', 'visitantes', 'conversiones', 'tasa_%']]
        .rename(columns={'tasa_%': 'tasa de conversión (%)'}))

# Guardamos los valores para usarlos después
vis_a  = resumen.loc[resumen['version'] == 'A', 'visitantes'].values[0]
vis_b  = resumen.loc[resumen['version'] == 'B', 'visitantes'].values[0]
conv_a = resumen.loc[resumen['version'] == 'A', 'conversiones'].values[0]
conv_b = resumen.loc[resumen['version'] == 'B', 'conversiones'].values[0]
tasa_a = resumen.loc[resumen['version'] == 'A', 'tasa'].values[0]
tasa_b = resumen.loc[resumen['version'] == 'B', 'tasa'].values[0]

print(f"Diferencia entre versiones: {tasa_a - tasa_b:+.2%}")
print(f"Mejora relativa de A sobre B: {(tasa_a - tasa_b) / tasa_b * 100:+.1f}%")


---
## Paso 3: Visualizar los resultados

Un buen analista siempre visualiza los datos antes de sacar conclusiones.  
Los gráficos nos ayudan a comunicar los resultados a personas no técnicas.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('A/B Test — La Redoma Rosa', fontsize=14, fontweight='bold')

colores = ['#d4537e', '#378add']

# ── Gráfico 1: Tasa de conversión ─────────────────────────────────────────
etiquetas = ['A — Vintage Rosa', 'B — Moderna']
tasas_pct = [tasa_a * 100, tasa_b * 100]

barras = axes[0].bar(etiquetas, tasas_pct, color=colores, width=0.45,
                     edgecolor='white', linewidth=1.5)

for barra, tasa in zip(barras, tasas_pct):
    axes[0].text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height() + 0.05,
        f'{tasa:.2f}%',
        ha='center', va='bottom', fontsize=12, fontweight='bold'
    )

axes[0].set_title('Tasa de conversión por versión', fontsize=12)
axes[0].set_ylabel('Conversiones (%)')
axes[0].set_ylim(0, max(tasas_pct) * 1.3)
axes[0].grid(axis='y', alpha=0.3)
axes[0].spines[['top', 'right']].set_visible(False)

# ── Gráfico 2: Conversiones vs. no conversiones ────────────────────────────
no_a = vis_a - conv_a
no_b = vis_b - conv_b

axes[1].bar([0, 1], [conv_a, conv_b], color=colores, label='Compró', width=0.45)
axes[1].bar([0, 1], [no_a, no_b], bottom=[conv_a, conv_b],
            color=['#f4c0d1', '#b5d4f4'], label='No compró', width=0.45)

axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(etiquetas, fontsize=10)
axes[1].set_title('Compradores vs. no compradores', fontsize=12)
axes[1].set_ylabel('Número de usuarios')
axes[1].legend(fontsize=10)
axes[1].grid(axis='y', alpha=0.3)
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('../imagenes/resultado_ab_test.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gráfico guardado como 'resultado_ab_test.png'")


### ¿Qué nos dicen estos gráficos?
**Gráfico izquierdo — Tasa de conversión por versión**     
Es el más importante. Muestra el porcentaje de visitantes que compraron en cada versión:

La Versión A (Vintage Rosa) convirtió aproximadamente el **6,35%** de sus visitantes
La Versión B (Moderna) convirtió aproximadamente el **4,10%** de sus visitantes

> 🫙 Estos valores pueden variar ligeramente según la semilla utilizada al generar el dataset.

A simple vista la A gana claramente, pero este gráfico solo nos dice que parece mejor. Para saber si la diferencia es real o podría ser azar necesitamos el test estadístico — que es el paso siguiente en el notebook.

**Gráfico derecho — Compradores vs. no compradores**     
Este pone los números en perspectiva. Cada barra representa los 4.000 usuarios de cada versión divididos en dos segmentos:

La parte oscura (abajo) son los que compraron
La parte clara (arriba) son los que no compraron

Lo que salta a la vista es que la inmensa mayoría no compra en ninguna de las dos versiones — las barras oscuras son muy pequeñas comparadas con el total. Eso es completamente normal en e-commerce: tasas de conversión del 4-7% son realistas y hasta buenas.
Este segundo gráfico es muy útil para comunicar resultados a personas no técnicas — de un vistazo entienden la magnitud real del fenómeno.

---
## Paso 4: Test estadístico

La versión A parece mejor... pero ¿es una diferencia real o podría ser pura casualidad?

Para responder esta pregunta usamos un **z-test de proporciones**.

### Las hipótesis

Antes de mirar el resultado, formulamos las hipótesis:

- **H₀ (hipótesis nula):** No hay diferencia entre las dos versiones. Cualquier diferencia observada se debe al azar.
- **H₁ (hipótesis alternativa):** La versión A convierte más que la versión B.

### El criterio de decisión

Si el **p-value es menor que 0.05**, rechazamos H₀ y concluimos que la diferencia es real (confianza del 95%).  
Si es mayor, no tenemos evidencia suficiente para declarar un ganador.


In [ ]:
# Ejecutamos el test estadístico
# alternative='larger' porque nuestra hipótesis es que A > B
conteos  = np.array([conv_a, conv_b])
muestras = np.array([vis_a,  vis_b])

z_stat, p_value = proportions_ztest(conteos, muestras, alternative='larger')

print("=== Resultado del test estadístico ===")
print(f"  Z-statistic : {z_stat:.4f}")
print(f"  P-value     : {p_value:.4f}")
print()

if p_value < 0.05:
    print(f"✅ p-value ({p_value:.4f}) < 0.05")
    print("   La diferencia ES estadísticamente significativa.")
    print("   Podemos rechazar H₀ con un 95% de confianza.")
else:
    print(f"⚠️  p-value ({p_value:.4f}) >= 0.05")
    print("   No hay evidencia suficiente para declarar un ganador.")


---
## Paso 5: Conclusión y decisión final


In [ ]:
lift = (tasa_a - tasa_b) / tasa_b * 100

print("=" * 52)
print("  INFORME FINAL — LA REDOMA ROSA")
print("=" * 52)
print()
print("EXPERIMENTO")
print("  ¿Qué estética web convierte mejor")
print("  para una marca de belleza natural en Asturias?")
print()
print("RESULTADOS")
print(f"  Versión A (Vintage Rosa) : {tasa_a:.2%} de conversión")
print(f"  Versión B (Moderna)      : {tasa_b:.2%} de conversión")
print(f"  Diferencia               : {tasa_a - tasa_b:+.2%}")
print(f"  Mejora de A sobre B      : +{lift:.1f}%")
print(f"  P-value                  : {p_value:.4f}")
print()
print("DECISIÓN")
if p_value < 0.05 and tasa_a > tasa_b:
    print("  ✅ La hipótesis se CONFIRMA.")
    print("  Implementar la Versión A (Vintage Rosa)")
    print("  al 100% de los usuarios.")
    print()
    print("  POR QUÉ GANÓ LA A:")
    print("  La estética vintage transmite autenticidad")
    print("  y confianza en los métodos tradicionales,")
    print("  coherente con la propuesta de valor de la")
    print("  marca. La estética moderna genera disonancia")
    print("  con el mensaje de naturalidad y artesanía.")
else:
    print("  Test INCONCLUSO.")
    print("  Recomendación: ampliar la muestra.")
print()
print("=" * 52)


---
## 🪮 Para reflexionar

1. ¿Qué habría pasado si hubiéramos parado el test al día 3 porque "parecía que ganaba la A"? ¿Qué error estaríamos cometiendo?

2. Fíjate en la columna `concejo`. ¿Crees que el resultado sería igual en Gijón que en Siero? ¿Cómo lo comprobarías?



In [ ]:
for concejo in ['Gijón', 'Siero']: # Un bucle que se ejecuta dos veces, una con Gijón y otra con Siero, podemos añadir al listado todos los concejos que queramos.
    sub = df[df['concejo'] == concejo] # Filtra el DataFrame para quedarse solo con las filas del concejo actual. Es como un filtro en Excel.
    print(f"\n=== {concejo} ===")
    print(sub.groupby('version')['convirtio'].mean().map('{:.2%}'.format))
    #.groupby('version') → agrupa las filas por versión (A y B).
    # ['convirtio'] → selecciona solo esa columna. 
    # .mean() → calcula la media de cada grupo — que al ser 0s y 1s equivale a la tasa de conversión.
    # .map('{:.2%}'.format) → formatea los números como porcentaje con 2 decimales.

3. El dataset también tiene `tiempo_sesion_seg`. ¿Los usuarios de la versión A pasan más tiempo en la web? ¿Eso es bueno o malo?

In [ ]:
print(df.groupby('version')['tiempo_sesion_seg'].mean().round(1))